# Stage 6 — Relative Karst Vulnerability / Risk (Mole Creek prototype)

The prototype's main output: combines Stage 4 (intrinsic karst
vulnerability) × Stage 5 (land-use pressure, 2020) into a single risk
surface — *"karst land-use pressure risk index"*, or *relative karst
vulnerability under land-use pressure*. Deliberately not called
"probability of contamination" or similar — this is a relative index,
not a calibrated prediction of actual damage.

Per the plan: once this looks geographically sensible, the method gets
**frozen** here before scaling statewide (Stage 7).

In [ ]:
from pathlib import Path
import os
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

data = Path(os.environ.get("AT3_DATA_DIR", "data"))
outpath = data / "Output_data"

with rasterio.open(outpath / "intrinsic_vulnerability_raw_mc_EPSG7855.tif") as src:
    intrinsic = src.read(1)
    intrinsic_nodata = src.nodata
    profile = src.profile
with rasterio.open(outpath / "landuse_pressure_mc_2020_EPSG7855.tif") as src:
    pressure = src.read(1).astype(float)
    pressure_nodata = src.nodata

valid = (intrinsic != intrinsic_nodata) & (pressure != pressure_nodata)

## 6.1 Combine

Risk = Intrinsic vulnerability × Land-use pressure (multiplicative,
consistent with Stage 4's own combination and COP's design). 2020 used
as the primary/current year — 2010 stays available for the Stage 9
temporal comparison, not duplicated here.

In [ ]:
pressure_norm = pressure / 3.0  # Stage 5's max pressure score
risk = np.where(valid, intrinsic * pressure_norm, np.nan)

present = np.unique(risk[valid])
print("Distinct risk values present:", present)

## 6.2 Classify

6 distinct values occur — a direct consequence of combining two
already-discrete inputs (same reasoning as Stage 4: with few underlying
combinations, classifying on the actual values is more meaningful than
imposing arbitrary quantile breaks). Not forced into exactly 5 labels
just to match a generic example scheme.

In [ ]:
sorted_vals = sorted(present)
LABELS = ["Very Low", "Low", "Moderate", "Moderate-High", "High", "Very High"][:len(sorted_vals)]

risk_class = np.full(risk.shape, 255, dtype="uint8")
for i, val in enumerate(sorted_vals):
    risk_class[valid & np.isclose(risk, val)] = i

print("Class distribution:")
for i, (val, label) in enumerate(zip(sorted_vals, LABELS)):
    n = (risk_class == i).sum()
    print(f"  {label} (score={val:.4f}): {n} cells ({n/valid.sum()*100:.1f}%)")

out_profile = profile.copy()
out_profile.update(dtype="uint8", nodata=255)
with rasterio.open(outpath / "risk_class_mc_2020_EPSG7855.tif", "w", **out_profile) as dst:
    dst.write(risk_class, 1)

risk_out = np.where(valid, risk, -9999).astype("float32")
raw_profile = profile.copy()
raw_profile.update(dtype="float32", nodata=-9999)
with rasterio.open(outpath / "risk_mc_2020_EPSG7855.tif", "w", **raw_profile) as dst:
    dst.write(risk_out, 1)

cmap = ListedColormap(["#f7f7f7", "#fee090", "#fdae61", "#f46d43", "#d73027", "#a50026"][:len(sorted_vals)])
display = np.ma.masked_equal(risk_class, 255)
plt.figure(figsize=(9, 9))
plt.imshow(display, cmap=cmap, vmin=0, vmax=len(sorted_vals) - 1)
cbar = plt.colorbar(ticks=range(len(sorted_vals)))
cbar.ax.set_yticklabels(LABELS)
plt.title("Mole Creek relative karst vulnerability under land-use pressure (2020)")
plt.show()

## 6.3 Does this make geographic sense?

Per the plan's own instruction before freezing the method: check the
result against something independent of the raster math. Mean risk by
named karst area (`KNAME`) is a natural check — if the flagship,
best-known, most intensely-karstified systems don't come out on top,
something's wrong.

In [ ]:
karst_mc = gpd.read_file(outpath / "karst_mc_EPSG7855.gpkg")
karst_mc = karst_mc[karst_mc.geom_type.isin(["Polygon", "MultiPolygon"])]
karst_mc = karst_mc[karst_mc["karst_intensity"] > 0].copy()  # actual karst only, not catchment polygons

with rasterio.open(outpath / "risk_mc_2020_EPSG7855.tif") as src:
    means = []
    for _, row in karst_mc.iterrows():
        out_image, _ = mask(src, [row.geometry], crop=True, nodata=-9999)
        vals = out_image[0]
        vals = vals[vals != -9999]
        means.append(vals.mean() if len(vals) else np.nan)
karst_mc["mean_risk"] = means

top = karst_mc.groupby("KNAME")["mean_risk"].mean().sort_values(ascending=False)
print("Named karst areas ranked by mean risk:")
print(top)

**Result: makes sense.** "Mole Creek" and "Mole Creek (Chudleigh Hill)"
— the flagship, most intensely-karstified, most cave-dense systems in
this area — come out on top, with peripheral/smaller karst areas
(Golden Valley, Quamby Brook) at the bottom. This is a genuine
plausibility confirmation, not a designed-to-pass check: nothing in the
combination logic explicitly favours these named areas — they rank
highest because their `KCATEGORY`/exposure/connectivity/land-use inputs
combine that way.

**Method is frozen from here.** Per the plan, Stage 7 applies this exact
same workflow statewide without re-tuning.

## Stage 6 outputs

- `risk_mc_2020_EPSG7855.tif` — continuous risk score
- `risk_class_mc_2020_EPSG7855.tif` — classified risk (labels above)

**Next (Stage 7):** apply the frozen method statewide at coarser
resolution.